# Fashion-MNIST Image Classification using EfficientNetB0 (Transfer Learning)
**Course:** ICE 4112 [Artificial Intelligence Lab] | **Framework:** TensorFlow / Keras

Steps: 1) Data Preparation  2) Model Implementation  3) Model Training  4) Performance Evaluation  5) Result Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 1. Data Preparation

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("Train:", x_train_full.shape, y_train_full.shape)
print("Test :", x_test.shape, y_test.shape)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_full[i], cmap='gray')
    plt.title(class_names[y_train_full[i]], fontsize=9)
    plt.axis('off')
plt.suptitle("Sample images from Fashion-MNIST")
plt.tight_layout()
plt.show()

In [ ]:
# Train / Validation split (stratified)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, stratify=y_train_full, random_state=SEED)

# Add channel dimension and keep pixel range 0-255
# (Keras EfficientNet has its own rescaling layer inside, so NO manual /255 here)
x_train = x_train[..., None].astype('float32')
x_val   = x_val[..., None].astype('float32')
x_test  = x_test[..., None].astype('float32')

print("Train:", x_train.shape, "| Val:", x_val.shape, "| Test:", x_test.shape)

BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(x, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(10000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(x_train, y_train, shuffle=True)
val_ds   = make_ds(x_val, y_val)
test_ds  = make_ds(x_test, y_test)

## 2. Model Implementation
- Fashion-MNIST images are 28x28 grayscale, but EfficientNet expects RGB. So we **resize to 96x96** and **repeat the channel 3 times** inside the model.
- Backbone: **EfficientNetB0** pretrained on ImageNet (`include_top=False`).
- New classification head: GlobalAveragePooling -> BatchNorm -> Dropout -> Dense(10, softmax).
- Data augmentation (flip, small rotation, zoom) is applied only during training.

In [ ]:
IMG_SIZE = 96
NUM_CLASSES = 10

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

base_model = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base_model.trainable = False   # Phase 1: freeze backbone

inputs = keras.Input(shape=(28, 28, 1))
x = layers.Resizing(IMG_SIZE, IMG_SIZE)(inputs)
x = layers.Concatenate()([x, x, x])           # gray -> 3 channels
x = data_augmentation(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="EfficientNetB0_FashionMNIST")
model.summary()

## 3. Model Training (Transfer Learning + Fine-tuning)
Overfitting reduction techniques used: **data augmentation, dropout, batch normalization, early stopping, learning-rate reduction, and a small fine-tuning learning rate.**

**Phase 1:** train only the new head (backbone frozen).  
**Phase 2:** unfreeze the top layers of EfficientNet and fine-tune with a very low learning rate.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint("best_effnet_fashion.keras", monitor="val_accuracy", save_best_only=True),
]

# ---------- Phase 1: feature extraction ----------
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

In [ ]:
# ---------- Phase 2: fine-tuning ----------
base_model.trainable = True
for layer in base_model.layers[:-30]:          # keep early layers frozen
    layer.trainable = False
for layer in base_model.layers:                # keep BatchNorm frozen (important for stability)
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history2 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

In [ ]:
# Training curves (both phases combined)
acc     = history1.history['accuracy']     + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss    = history1.history['loss']         + history2.history['loss']
val_loss= history1.history['val_loss']     + history2.history['val_loss']
split = len(history1.history['accuracy'])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(acc, label='Train'); ax[0].plot(val_acc, label='Validation')
ax[0].axvline(split - 1, color='gray', linestyle='--', label='Fine-tune start')
ax[0].set_title('Accuracy'); ax[0].set_xlabel('Epoch'); ax[0].legend()
ax[1].plot(loss, label='Train'); ax[1].plot(val_loss, label='Validation')
ax[1].axvline(split - 1, color='gray', linestyle='--', label='Fine-tune start')
ax[1].set_title('Loss'); ax[1].set_xlabel('Epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Performance Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification Report (Precision / Recall / F1-score):\n")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix - Fashion-MNIST (EfficientNetB0)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Per-class accuracy
per_class = cm.diagonal() / cm.sum(axis=1)
for name, a in zip(class_names, per_class):
    print(f"{name:12s}: {a:.4f}")

# Most confused pairs
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
print("\nTop confusions:")
for idx in np.dstack(np.unravel_index(np.argsort(-cm_off.ravel())[:5], cm_off.shape))[0]:
    t, p = idx
    print(f"  True '{class_names[t]}' -> Predicted '{class_names[p]}' : {cm_off[t, p]} times")

## 5. Result Analysis

In [ ]:
# Visualize predictions (green = correct, red = wrong)
idxs = np.random.choice(len(x_test), 15, replace=False)
plt.figure(figsize=(12, 7))
for i, idx in enumerate(idxs):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[idx].squeeze(), cmap='gray')
    color = 'green' if y_pred[idx] == y_test[idx] else 'red'
    plt.title(f"T: {class_names[y_test[idx]]}\nP: {class_names[y_pred[idx]]}", color=color, fontsize=8)
    plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Misclassified examples
wrong = np.where(y_pred != y_test)[0]
print("Total misclassified:", len(wrong), "out of", len(y_test))
plt.figure(figsize=(12, 4))
for i, idx in enumerate(wrong[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx].squeeze(), cmap='gray')
    plt.title(f"T: {class_names[y_test[idx]]}\nP: {class_names[y_pred[idx]]}", fontsize=8, color='red')
    plt.axis('off')
plt.tight_layout(); plt.show()

### Observations (edit with your own numbers after running)
- EfficientNetB0 pretrained on ImageNet was adapted to 28x28 grayscale Fashion-MNIST by resizing to 96x96 and replicating the channel.
- Transfer learning (frozen backbone) gave a strong baseline quickly; fine-tuning the top layers with a low learning rate improved accuracy further.
- Overfitting was controlled using augmentation, dropout, batch normalization, early stopping and learning-rate reduction; the train/validation curves stay close.
- Classes like **Trouser, Bag, Sandal, Sneaker, Ankle boot** are classified with very high precision/recall.
- Most confusion occurs among visually similar upper-body garments: **Shirt vs T-shirt/top vs Pullover vs Coat**. Shirt has the lowest F1-score.
- Possible improvements: larger input size, EfficientNetB1/B2, more fine-tuning epochs, stronger augmentation, ensembling.